In [ ]:
import os
from dotenv import load_dotenv

# ──────────────────────────────────────────────
# 환경변수
# ──────────────────────────────────────────────
load_dotenv(override = True)

# OPENROUTER SET
OPENROUTER_API_KEY  = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_URL      = "https://openrouter.ai/api/v1"
# MODEL_NAME          = "openai/gpt-oss-120b:free"
# MODEL_NAME          = "nvidia/nemotron-3-super-120b-a12b:free"

# ELICE SET
ELICE_API_KEY       = os.getenv("ELICE_API_KEY")
ELICE_URL           = os.getenv("ELICE_URL")

MODEL_NAME          = "zai-org/glm-4.7"

# OTHER TOOLS
OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")

# LangChain의 주요 구성 요소 탐색

### 1. LLM

- LLM 객체를 살펴봅니다.

### 2. Prompt template

- LLM의 output generation에 영향을 주는 prompt template를 살펴봅니다.

### 3. LangChain Expression Language (LCEL)

- 다양한 컴포넌트를 결합하여 하나의 아키텍쳐를 구성할 수 있도록 하는 LangChain 언어를 살펴봅니다.

### 4. 이외 구성 요소

- 위 세 가지 구성 요소 이외 중요한 컴포넌트들이 있습니다. 간단하게만 언급하고 이후 강의에서 자세히 다룰 예정입니다.

## 1. LLM

- <span style="background-color: #A5B68D; border-radius: 5px; padding: 2px 6px; border: 1px solid #ccc; font-family: sans-serif;">
    LLM</span> : LLM 모델
    
    - `invoke` : LLM 호출
    - `batch` : 여러개의 인풋을 독립적으로 호출 (parallel processing)
    - `stream` : Output streaming
    - `ainvoke` / `abatch` / `astream`

In [ ]:

from langchain_openai import ChatOpenAI

# 모델 초기화
llm = ChatOpenAI(
    model    = MODEL_NAME,
    api_key  = ELICE_API_KEY,
    base_url = ELICE_URL,
)

In [5]:
output = llm.invoke("HI")

In [7]:
output = llm.batch(["HI", 
                    "My name is Gieun, What is your name?"])

print(output)

[AIMessage(content="Hello! I'm GLM, trained by Z.ai. How can I assist you today? Whether you have questions or just want to chat, I'm happy to help.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 6, 'total_tokens': 42, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'reasoning_tokens': 0}, 'model_provider': 'openai', 'model_name': 'glm-4.7-fp8', 'system_fingerprint': None, 'id': 'bc7da305504c4670baa58617b598e374', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e155a-5fba-79c3-8836-49a8246ea005-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 6, 'output_tokens': 36, 'total_tokens': 42, 'input_token_details': {}, 'output_token_details': {}}), AIMessage(content="Hello Gieun! My name is GLM, and I'm a large language model trained by Z.ai. It's nice to meet you! 😊\n\nHow can I help you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'co

In [10]:
answers = list(map(lambda x: x.text, output))

for answer in answers:
    print(answer)

Hello! I'm GLM, trained by Z.ai. How can I assist you today? Whether you have questions or just want to chat, I'm happy to help.
Hello Gieun! My name is GLM, and I'm a large language model trained by Z.ai. It's nice to meet you! 😊

How can I help you today?


## 2. Prompt template

- <span style="background-color: #A5B68D; border-radius: 5px; padding: 2px 6px; border: 1px solid #ccc; font-family: sans-serif;">
    PromptTemplate</span> : LLM 인풋을 위한 Prompt 객체
    
    - `from_template` : Prompt를 구체적으로 정의

In [11]:
from langchain_core.prompts import PromptTemplate

# template를 정의
template = "{movie_name}에 나오는 등장인물을 알려줘."

# from_template : Langchain의 promptTemplate 객체를 생성
prompt = PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['movie_name'], input_types={}, partial_variables={}, template='{movie_name}에 나오는 등장인물을 알려줘.')

In [12]:
# .format을 사용하여 prompt를 dynamic하게 완성할 수 있음
prompt.format(movie_name="반지의 제왕")

'반지의 제왕에 나오는 등장인물을 알려줘.'

여러 개의 input을 prompt에 추가할 수 있습니다.

In [14]:
template = "{movie_name}에 나오는 등장인물을 알려줘. 그리고 {menu_name}에 들어가는 재료도 알려줘."

# from_template : Langchain의 promptTemplate 객체를 생성
prompt = PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['menu_name', 'movie_name'], input_types={}, partial_variables={}, template='{movie_name}에 나오는 등장인물을 알려줘. 그리고 {menu_name}에 들어가는 재료도 알려줘.')

In [15]:
prompt.format(movie_name="반지의 제왕", menu_name='김치찌개')

'반지의 제왕에 나오는 등장인물을 알려줘. 그리고 김치찌개에 들어가는 재료도 알려줘.'

In [16]:
llm.invoke(prompt.format(movie_name="반지의 제왕", menu_name='김치찌개'))

AIMessage(content="두 가지 전혀 다른 주제를 질문해 주셨네요! **반지의 제왕**의 주요 등장인물과 맛있는 **김치찌개**의 재료를 정리해 드릴게요.\n\n---\n\n### 1. 반지의 제왕 (The Lord of the Rings) 주요 등장인물\n\nJ.R.R. 톨킨의 판타지 소설 및 영화 시리즈에 나오는 핵심 인물들입니다.\n\n**[호빗 (The Hobbits)]**\n*   **프로도 배긴스 (Frodo Baggines):** 절대반지를 파괴하기 위해 여정을 떠난 주인공.\n*   **샘 와이즈 감지 (Samwise Gamgee):** 프로도의 충실한 정원사이자 가장 친한 친구.\n*   **메리아독 브랜디벅 (Merry) & 페레그린 툭 (Pippin):** 프로도와 함께 여정에 동참한 또 다른 호빗 친구들.\n\n**[마법사 (The Wizards)]**\n*   **간달프 (Gandalf):** 회색 마법사이자 반지 원정대의 지도자. 후에 백색 마법사로 다시 태어납니다.\n*   **사루만 (Saruman):** 백색 마법사였으나 타락하여 사우론과 손잡고 적이 됩니다.\n\n**[인간 (Men)]**\n*   **아라곤 (Aragorn):** 곤도르의 왕位 계승자이며 북방의 방랑자. 뛰어난 전사입니다.\n*   **보로미르 (Boromir):** 곤도르의 대장군이며 센스게스왕의 아들. 반지의 유혹에 흔들리나 용감하게 싸웁니다.\n*   **테오덴 (Theoden):** 로한의 왕.\n*   **에오윈 (Eowyn):** 로한의 공주이며 여성 전사.\n\n**[엘프 (Elves)]**\n*   **레골라스 (Legolas):** 순궁(요정 궁수)으로, 눈이 예민하고 활쏘기에 도가 텄습니다.\n*   **갈라드리엘 (Galadriel):** 로스로리엔의 여왕이자 고위 엘프.\n*   **엘롯드 (Elrond):** 리븐델의 주인.\n\n**[드워프 (Dwarf)]**\n*   **김리 (Gimli):** 건장한 드워프 전사로, 도끼

`input_types`, `input_variables` 등 다음과 같이 PromptTemplate의 input을 구체적으로 정의할 수 있습니다.


In [17]:
template = "{movie_name}에 나오는 등장인물을 알려줘. 그리고 {menu_name}에 들어가는 재료도 알려줘."

prompt = PromptTemplate(
    template=template,
    input_variables=["movie_name", "menu_name"],
    input_types={"movie_name":str, "menu_name": str}
)

prompt.format(movie_name=1, menu_name=2)

'1에 나오는 등장인물을 알려줘. 그리고 2에 들어가는 재료도 알려줘.'

- <span style="background-color: #A5B68D; border-radius: 5px; padding: 2px 6px; border: 1px solid #ccc; font-family: sans-serif;">
    ChatPromptTemplate</span> : Prompt를 대화형으로 제공합니다.
    
    - `from_template` : Prompt를 구체적으로 정의
    - `from_messages` : role, message를 대화형으로 추가하여 순차적인 prompt 생성 가능

In [18]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_template("{movie_name}에 나오는 등장인물을 알려줘.")
# HumanMessagePromptTemplate가 생긴 것을 볼 수 있음
chat_prompt

ChatPromptTemplate(input_variables=['movie_name'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['movie_name'], input_types={}, partial_variables={}, template='{movie_name}에 나오는 등장인물을 알려줘.'), additional_kwargs={})])

In [36]:
# role, message 형태로 대화를 제공
chat_template = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 영화 전문가입니다. 하지만 {movie_name}에 관해서만 대화를 하고싶어하며, 다른 영화가 나오면 대화를 거부합니다."),
        ("human", "안녕하세요"),
        ("assistant", "안녕하세요! 무엇을 도와드릴까요?"),
        ("user", "{human_input}"),
    ]
)
# role은 'human', 'user', 'ai', 'assistant', 또는 'system'만 가능
# ai=assistant, human=user, system
# 각각의 prompt 형식에 맞게끔 prompt 구축됨

chat_history = chat_template.format_messages(
    # movie_name="반지의 제왕", human_input="해리포터의 등장인물을 알려줘"
    movie_name="백투더퓨쳐", human_input="백투더퓨쳐의 등장인물을 알려줘"
)

In [37]:
response = llm.invoke(chat_history)


In [38]:
print(response.content)

물론이죠! '백 투 더 퓨처' 시리즈의 핵심 등장인물들을 소개해 드릴게요.

1.  **마티 맥플라이 (Marty McFly):** 마이클 J. 폭스가 연기한 주인공으로, 평범한 고등학생이지만 타임머신 덕분에 시간 여행을 하게 되죠. 기타를 치는 걸 좋아하고 스케이트보드를 잘 탑니다.
2.  **닥터 에멧 브라운 (Dr. Emmett "Doc" Brown):** 크리스토퍼 로이드가 연기한 광기 어린 발명가입니다. 델리언 DMC-12를 개조하여 타임머신을 만들었죠. 그의 말버릇인 "Great Scott!"은 유명합니다.
3.  **로레인 베인스 맥플라이 (Lorraine Baines McFly):** 마티의 어머니입니다. 1955년으로 시간 여행을 한 마티가 우연히 만나게 되면서 사건이 벌어지죠.
4.  **조지 맥플라이 (George McFly):** 마티의 아버지로 소심하고 내성적인 성격입니다. 과거에서 마티의 도움을 받아 자신감을 갖게 되죠.
5.  **비프 태넌 (Biff Tannen):** 시리즈 내내 마티와 그의 가족을 괴롭히는 악당입니다. 특히 조지를 괴롭히는 불량 배 역할로 등장합니다.
6.  **제니퍼 파커 (Jennifer Parker):** 마티의 여자친구입니다.

외에도도 스탠더드(Principal Strickland) 교장 같은 인물들도 중요한 역할을 하죠. 이 영화의 캐릭터들은 정말 임팩트가 강하지 않나요?


## 3. LangChain Expression Language (LCEL)

LangChain의 컴포넌트들을 블럭처럼 연결할 수 있습니다.

`component1 | component2 | component3`의 형태로 구현이 되며, 이전 component의 output을 다음의 인풋으로 전달하는 방식입니다.

다양한 컴포넌트들을 유기적으로 연결하여 하나의 객체를 만들 수 있습니다.

In [23]:
template = "{movie_name}에 나오는 등장인물을 알려줘. 그리고 {menu_name}에 들어가는 재료도 알려줘."

prompt = PromptTemplate(
    template=template,
    input_variables=["movie_name", "menu_name"],
    input_types={"movie_name":str, "menu_name": str}
)

prompt.format(movie_name="반지의 제왕", menu_name="불고기")

'반지의 제왕에 나오는 등장인물을 알려줘. 그리고 불고기에 들어가는 재료도 알려줘.'

In [24]:
chain = prompt | llm

In [25]:
chain.invoke({"movie_name":"반지의 제왕", "menu_name":"불고기"})

AIMessage(content="반지의 제왕과 불고기라는 아주 재미있는 조합을 질문해 주셨네요! 각각에 대해 알기 쉽게 정리해 드릴게요.\n\n### 1. 반지의 제왕 주요 등장인물\n\nJ.R.R. 톨킨의 원작과 영화 삼부작 모두에서 중요한 역할을 하는 인물들입니다.\n\n**[호빗과 반지 원정대]**\n*   **프로도 배긴스:** 주인공. 절대 반지를 운명의 산(운도레)으로 가져가는 사명을 띤 호빗입니다.\n*   **samwise gamgee (샘):** 프로도의 충실한 정원사이자 친구. 프로도를 끝까지 보호하는 든든한 조력자입니다.\n*   **간달프:** 회색 마법사이자 백색 마법사. 원정대를 이끄는 지혜로운 스승 같은 존재입니다.\n*   **아라곤:** 북방의 방랑자이며 곤도르의 정통 왕위 계승자. 검과 지략을 겸비한 영웅입니다.\n*   **레골라스:** 요정(엘프) 궁수. 시력이 좋고 활솜씨가 뛰어납니다.\n*   **김리:** 드워프 전사. 도끼를 잘 쓰며, 레골라스와는 시작은 티격태격했지만 나중에는 절친이 됩니다.\n*   **보로미르:** 곤도르의 섭정 아들의 장남. 인간의 약한 면모와 용기를 동시에 보여주는 인물입니다.\n*   **메리와 피핀:** 프로도의 호빗 친구들입니다. 엉뚱하지만 전쟁에서 큰 역할을 합니다.\n\n**[적대 세력 및 기타 중요 인물]**\n*   **사우론:** 절대 반지를 만든 암흑의 주인. 모르도르를 지배합니다.\n*   **고램 (스미골):** 반지를 '내 보물'이라 부르며 집착하는 불쌍한 존재입니다.\n*   **아르웬:** 륀실의 딸인 요정 공주이며 아라곤의 연인입니다.\n*   **사루만:** 백색 마법사였으나 사우론에게 타락하여 적이 됩니다.\n*   **셀로브림:** 로스리엔의 숲을 다스리는 요정 여왕입니다.\n\n---\n\n### 2. 불고기에 들어가는 재료\n\n집에서 쉽게 만드는 **한식 불고기** 기준으로 알려드릴게요.\n\n**[주재료]**\n*   **소고기(우둔살, 국거리, 등심,

## 4. 이외 구성 요소


In [ ]:

# Chat에 과거 히스토리를 추가하여 과거 질의 내용도 함께 고려 할 수 있습니다.
# 메모리는 대화의 맥락을 유지하는 데 사용됩니다. 대화형 에이전트가 이전 대화 내용을 기억하고 활용할 수 있도록 도와줍니다.

from langchain_core.document_loaders.base import BaseLoader
# 다양한 파일이나 api 등 외부 소스와 연결하여 정보를 취사선택 후 LLM에게 넘겨줄 수 있습니다.

from langchain_community.document_loaders.news import NewsURLLoader
# NewsURLLoader는 뉴스 기사 URL을 로드하여 해당 뉴스 기사의 내용을 LLM에게 전달할 수 있도록 도와주는 도구입니다.

In [29]:
import langchain
print(langchain.__version__)
print(langchain.__file__)

1.1.3
c:\Users\rkfka\workspace\FC_Agent_bible\.venv\Lib\site-packages\langchain\__init__.py


In [ ]:
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI

# 1. 모델 설정
model = ChatOpenAI(
    model    = MODEL_NAME,
    api_key  = ELICE_API_KEY,
    base_url = ELICE_URL,
)

# 2. 세션별로 메시지를 저장할 딕셔너리
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# 3. 메모리가 결합된 체인 생성
with_message_history = RunnableWithMessageHistory(
    model,
    get_session_history,
)

# 4. 사용 예시 (config에 session_id 전달)
config = {"configurable": {"session_id": "user_123"}}
response = with_message_history.invoke(
    [HumanMessage(content="내 이름은 철수야")], 
    config = config
)

In [40]:
response

AIMessage(content='반가워요, 철수님! 오늘 어떤 도움이 필요하신가요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 12, 'total_tokens': 37, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'reasoning_tokens': 0}, 'model_provider': 'openai', 'model_name': 'glm-4.7-fp8', 'system_fingerprint': None, 'id': '3c8753a493cb464eac7fcb489a3104b4', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e156d-b26c-7ac3-bcea-560a40de0c15-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 25, 'total_tokens': 37, 'input_token_details': {}, 'output_token_details': {}})

In [41]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

# 1. 모델 설정
model = ChatOpenAI(
    model    = MODEL_NAME,
    api_key  = ELICE_API_KEY,
    base_url = ELICE_URL,
)

# 2. 프롬프트 구성 (메모리가 들어갈 공간인 MessagesPlaceholder가 핵심)
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 친절한 어시스턴트야."),
    MessagesPlaceholder(variable_name="history"), # 대화 기록이 주입될 위치
    ("human", "{question}")                      # 사용자 질문
])

# 3. 체인 생성 (프롬프트 | 모델)
chain = prompt | model

# 4. 세션별 메시지 저장소 설정
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# 5. 메모리가 결합된 최종 Runnable 생성
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="question",    # 사용자의 입력 변수 명
    history_messages_key="history",   # 프롬프트의 기록 변수 명
)

# 6. 실행 예시
config = {"configurable": {"session_id": "user_abc_123"}}

# 첫 번째 대화
response1 = with_message_history.invoke(
    {"question": "안녕, 내 이름은 제이슨이야."}, 
    config=config
)
print(f"AI: {response1.content}")

# 두 번째 대화 (이름을 기억하는지 확인)
response2 = with_message_history.invoke(
    {"question": "내 이름이 뭐라고?"}, 
    config=config
)
print(f"AI: {response2.content}")


AI: 안녕하세요, 제이슨 님! 만나서 반가워요. 오늘 어떤 도움이 필요하신가요?
AI: 제이슨 님이시죠!
